Note: 

Both SwapRateMCSimulator and SwapRatePCAEstimator require historical swap rate time series data in a specific format.
If no explicit file path is provided to the constructor, a default CSV file will be used instead.

The default dataset is located in the "curve_data" folder of the project (see "eur_swap_ts.csv" for the expected format).

In the examples below, a file path is explicitly provided, although it points to the same data as the default.

This notebook demonstrates:
- MC simulation of EUR swap curve. The EUR swap curve is calibrated to simulated swap rates. This transforms simulated swap rates to a set of derived
  EUR swap discounting curves which can be used to generate PnL scenarios and calculate CVaR.  

In [12]:
from __future__ import annotations
import os 
from pathlib import Path

# It is assumed that the notebook is executed from within the project - either root or sub folder. 
CWD_PATH = Path.cwd()
SWAP_TIME_SERIES_CSV_PATH = ""

for parent in CWD_PATH.parents:
 if (parent / "src").exists():
     SWAP_TIME_SERIES_CSV_PATH = parent / "curve_data" / "eur_swap_ts.csv"
     break       
 else:
     SWAP_TIME_SERIES_CSV_PATH = CWD_PATH / "curve_data" / "eur_swap_ts.csv"

print(CWD_PATH)
print(SWAP_TIME_SERIES_CSV_PATH)   

C:\Users\mduba\Desktop\Programming\Python\alm_ir_hedging_prototype\example_notebooks
C:\Users\mduba\Desktop\Programming\Python\alm_ir_hedging_prototype\curve_data\eur_swap_ts.csv


In [13]:
# imports 
from typing import Dict
from pathlib import Path
from datetime import datetime 
import numpy as np
import pandas as pd 
from curve_engine.swap_rate_pca_estimator import SwapRatePCAEstimator
from curve_engine.swap_rate_mc_simulator import SwapRateMCSimulator
from curve_engine.swap_curve_scenario_generator import SwapCurveScenarioGenerator

In [14]:
# parameters 
curve_date = datetime(2026,3,31)

In [15]:
# run pca from csv file with time series data for swap rates
swap_rate_pca = SwapRatePCAEstimator.from_csv( 
 swap_ts_csv_path = SWAP_TIME_SERIES_CSV_PATH 
)

In [16]:
# run mc simulation with scenarios for market/swap rates - 1000 scenarios, 1d shocks scaled to 30 day horizon
swap_rate_sim = SwapRateMCSimulator.from_csv(
  n_scenarios = 1000, 
  t_scale_factor = 30, 
  rng_seed = 1, 
  swap_ts_csv_path = SWAP_TIME_SERIES_CSV_PATH 
)

In [17]:
# create scenarios for curves calibrated to mc simulated market/swap rates 
scenarios = SwapCurveScenarioGenerator( 
    curve_date = curve_date, 
    swap_rate_sim = swap_rate_sim, 
    swap_rate_pca = swap_rate_pca
) 

In [18]:
# calibrate the curves in each scenario of swap rates
# Note! For 1000 scenarios, this takes approx. 5-6 mins. 
scenarios.build_mc_scenario_curves( curve_id = 'eurswap' )

SUCCESS: `func_tol` reached after 7 iterations (levenberg_marquardt), `f_val`: 3.13574640726366e-16, `time`: 0.1001s
SUCCESS: `func_tol` reached after 7 iterations (levenberg_marquardt), `f_val`: 6.50531716466949e-16, `time`: 0.0940s
SUCCESS: `func_tol` reached after 7 iterations (levenberg_marquardt), `f_val`: 5.137182868220651e-16, `time`: 0.0992s
SUCCESS: `func_tol` reached after 7 iterations (levenberg_marquardt), `f_val`: 1.6703745712172888e-16, `time`: 0.0918s
SUCCESS: `func_tol` reached after 7 iterations (levenberg_marquardt), `f_val`: 9.661113896766674e-16, `time`: 0.0981s
SUCCESS: `func_tol` reached after 7 iterations (levenberg_marquardt), `f_val`: 3.4509528820842696e-16, `time`: 0.0963s
SUCCESS: `func_tol` reached after 7 iterations (levenberg_marquardt), `f_val`: 7.586365078553314e-16, `time`: 0.1026s
SUCCESS: `func_tol` reached after 7 iterations (levenberg_marquardt), `f_val`: 3.474079273925337e-16, `time`: 0.0905s
SUCCESS: `func_tol` reached after 7 iterations (levenber

In [11]:
# mc curve scenario attributes 
scenarios.curve_date
scenarios.mc_n_scenarios
scenarios.mc_t_scale_factor
scenarios.hist_rates
scenarios.hist_shifts
scenarios.hist_cshifts
scenarios.hist_cov_mat 
scenarios.hist_corr_mat
scenarios.sim_rates
scenarios.sim_shifts
scenarios.mc_scenarios    
scenarios.mc_scenario_curves    
scenarios.pca_eigvals
scenarios.pca_eigvecs
scenarios.pca_loadings
scenarios.sim_pc_multipliers

,Scenario,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10
0,1,42.159106,2.414514,-8.328188,-0.569149,1.875678,0.514132,1.558732,-1.196780,-0.433708,-4.026122
1,2,-26.714762,-20.041426,-4.324316,-0.093540,4.881759,-2.071991,-0.848954,2.904716,0.753696,0.187443
2,3,2.687222,-12.474451,22.354319,4.568611,0.083991,-7.566429,3.270098,-7.135585,-0.857718,4.490638
3,4,142.152473,-12.735915,4.640567,-2.404433,-2.395499,9.599881,2.670557,6.864854,-1.794660,3.940224
4,5,-99.931880,8.880176,4.696588,2.344046,1.214362,5.894763,2.680165,-2.317420,1.026530,-1.083055
...,...,...,...,...,...,...,...,...,...,...,...
995,996,64.001480,19.000070,-4.305759,-0.912209,-8.466307,8.417770,0.660693,-4.025888,0.607281,-1.156745
996,997,-160.084397,-7.603759,-6.696594,0.386591,21.622697,3.066450,3.179821,-2.195182,1.422898,5.635364
997,998,-2.175155,-1.605786,-13.935167,3.220912,8.235920,-3.617492,1.589147,-5.921688,-0.727349,-1.019527
998,999,17.161090,14.809326,7.325375,-2.869595,0.252579,3.024897,-3.560887,1.569728,-1.580331,-2.051777
